# index-by-tensor — worked example 3: Paired versus grid indexing with two index tensors

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `index-by-tensor`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

Two index tensors of the same shape index element-wise (paired): `x[rows, cols]` collapses to one value per pair. Adding a new axis to each via `[:, None]` and `[None, :]` broadcasts them into a Cartesian grid: `x[rows[:, None], cols[None, :]]` returns every (row, col) combination.

## Worked solution

We extract the same data in two layouts using two index tensors.

1. `x` is `(H, W)`; `rows` and `cols` are 1-D `LongTensor`s of length `K`.
2. Paired: `x[rows, cols]` zips the two index tensors, so element `i` is `x[rows[i], cols[i]]`. The result is `(K,)` — the shared index shape.
3. Grid: reshape `rows` to `(K, 1)` with `rows[:, None]` and `cols` to `(1, K)` with `cols[None, :]`. Advanced indexing broadcasts these against each other, giving `(K, K)` where entry `[i, j]` is `x[rows[i], cols[j]]`.
4. The broadcasting of the two index tensors' shapes is what determines paired versus grid output. We verify a paired entry and a grid entry against direct scalar indexing.

In [ ]:
import torch as t

t.manual_seed(2)
x = t.randn(5, 6)
rows = t.tensor([0, 2, 4])
cols = t.tensor([1, 3, 5])

def paired_and_grid(x, rows, cols):
    return {
        'paired': x[rows, cols],
        'grid': x[rows[:, None], cols[None, :]],
    }

res = paired_and_grid(x, rows, cols)
print(res['paired'].shape, res['grid'].shape)
print('paired[1] == x[2,3]:', bool(res['paired'][1] == x[2, 3]))
print('grid[0,2] == x[0,5]:', bool(res['grid'][0, 2] == x[0, 5]))